# 02c — Qualitative & Financial Analysis

The Phase-2 analysis gate. Before any hybrid tuning or final test-set run, it puts every prompt variant through two checks: a business-cost ledger (what each variant's error profile would cost a portfolio) and a GPT-5.4 reasoning fingerprint (how each one actually reasons). A weak or over-speculative design gets caught here rather than downstream.

## Portfolio Simulation Controls
Adjust loan volume and risk parameters here. All ledger calculations below update dynamically.

In [1]:
# ==========================================
# BANC SABADELL PORTFOLIO SIMULATION CONTROLS
# ==========================================
PORTFOLIO_SIZE = 1000          # Number of credit applications to score
DEFAULT_RATE = 0.15           # Expected default rate (15.0%)
AVERAGE_LOAN_AMOUNT = 10000    # Average loan size in USD/EUR
LGD = 0.50                    # Expected Loss Given Default (50% lost on default)
EXPECTED_PROFIT = 2000         # Expected interest profit per repaid loan (USD/EUR)
# ==========================================
print(f"Simulation parameters set: Portfolio={PORTFOLIO_SIZE}, Default Rate={DEFAULT_RATE*100}%, Loan Size={AVERAGE_LOAN_AMOUNT}")

Simulation parameters set: Portfolio=1000, Default Rate=15.0%, Loan Size=10000


### The cost model

One function converts any variant's metrics into a portfolio P&L: missed defaults cost `LGD × loan`, false rejections cost the foregone `EXPECTED_PROFIT`, and API tokens are priced with the prompt-caching discount. Baselines (XGBoost, the 01a base prompt) are injected so every variant is judged against the same reference.

In [2]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from llm_utils import call_llm, load_api_key

DATA_DIR = "../../../data"
RESULTS_DIR = "../../../data/results"
PREDS_2B_PATH = "../../../data/results/llm/02b_predictions.csv"
METRICS_2B_PATH = "../../../data/results/llm/02b_phase1_metrics.csv"
LLM_CALLS_PATH = "../../../data/results/llm/llm_calls.csv"

# Load OpenAI API keys from .env for key-rotation and parallel execution
load_api_key("openai")
_key1 = os.environ.get("OPENAI_API_KEY")
_key2 = os.environ.get("OPENAI_API_KEY_2") or _key1
_key3 = os.environ.get("OPENAI_API_KEY_3") or _key1

API_KEYS = [_key1, _key2, _key3]
API_KEYS = [k for k in API_KEYS if k]
assert API_KEYS, "Set OPENAI_API_KEY in notebooks/llm_models/.env"

def run_financial_simulation(metrics_df_path, baseline_condition_name=None):
    metrics_df = pd.read_csv(metrics_df_path)
    _calls = pd.read_csv(LLM_CALLS_PATH) if os.path.exists(LLM_CALLS_PATH) else pd.DataFrame()
    
    # Inject baseline rows (both XGBoost and GPT-5.4 baseline prompt) from Phase 1a
    new_rows = []
    baseline_path = "../../../data/results/llm/01a_metrics.csv"
    if os.path.exists(baseline_path):
        df_01a = pd.read_csv(baseline_path)
        
        # 1. Find XGBoost
        sub_xgb = df_01a[df_01a['model'] == 'XGBoost']
        if not sub_xgb.empty:
            xgb_row = sub_xgb.iloc[0]
            new_rows.append({
                'variant': 'XGBoost (structured)',
                'accuracy': float(xgb_row['accuracy']),
                'precision_charged_off': float(xgb_row['precision_charged_off']),
                'recall_charged_off': float(xgb_row['recall_charged_off']),
                'f1_charged_off': float(xgb_row['f1_charged_off']),
                'auc': float(xgb_row['auc']) if pd.notna(xgb_row['auc']) else np.nan,
                'n_valid': int(xgb_row['n_valid'])
            })
            
        # 2. Find GPT-5.4 no_desc (baseline prompt)
        sub_base = df_01a[(df_01a['model'] == 'GPT-5.4') & (df_01a['condition'] == 'no_desc')]
        if not sub_base.empty:
            base_row = sub_base.iloc[0]
            new_rows.append({
                'variant': 'baseline',
                'accuracy': float(base_row['accuracy']),
                'precision_charged_off': float(base_row['precision_charged_off']),
                'recall_charged_off': float(base_row['recall_charged_off']),
                'f1_charged_off': float(base_row['f1_charged_off']),
                'auc': float(base_row['auc']) if pd.notna(base_row['auc']) else np.nan,
                'n_valid': int(base_row['n_valid'])
            })
            
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        new_df = new_df[~new_df['variant'].isin(metrics_df['variant'].values)]
        metrics_df = pd.concat([new_df, metrics_df], ignore_index=True)
        print(f"Injected baselines from Phase 1a: {new_df['variant'].tolist()}")
    
    defaults = PORTFOLIO_SIZE * DEFAULT_RATE
    good_loans = PORTFOLIO_SIZE * (1.0 - DEFAULT_RATE)
    lgd_cost = AVERAGE_LOAN_AMOUNT * LGD
    
    results = []
    for _, row in metrics_df.iterrows():
        condition = str(row.get('variant', row.iloc[0]))
        
        recall = row.get('recall_charged_off', np.nan)
        if pd.isna(recall):
            recall = row.get('recall', 0.0)
            
        precision = row.get('precision_charged_off', np.nan)
        if pd.isna(precision):
            precision = row.get('precision', 0.0)
            
        cost_mean = row.get('cost_mean', np.nan)
        if pd.isna(cost_mean):
            sub = pd.DataFrame()  # Safe initialization to avoid UnboundLocalError
            if "XGBoost" in condition:
                cost_mean = 0.0
            elif condition == "baseline":
                # Pull baseline cost from Phase 1b consistency runs
                sub = _calls[
                    (_calls['label'].str.contains("GPT-5.4 Run", na=False)) &
                    (_calls['desc_tag'] == "no_desc") &
                    (_calls['notebook_id'] == '01b_Consistency.ipynb')
                ] if not _calls.empty else _calls
            else:
                sub = _calls[
                    (_calls['label'] == f"GPT-5.4 | {condition}") &
                    (_calls['notebook_id'].isin(['02_prompt_variance', '02b_Prompt_Variance.ipynb']))
                ] if not _calls.empty else _calls
                
            if not sub.empty:
                cached_costs = []
                for _, call_row in sub.iterrows():
                    in_t = call_row.get('input_tokens', 0)
                    out_t = call_row.get('output_tokens', 0)
                    p_in = call_row.get('input_price_per_1k_usd', 0)
                    p_out = call_row.get('output_price_per_1k_usd', 0)
                    
                    if "few_shot" in condition or "few-shot" in condition:
                        c = 1600
                    elif in_t > 1024:
                        c = 200
                    else:
                        c = 0
                        
                    if in_t > 1024 and c > 0:
                        cached_t = min(in_t, c)
                        uncached_t = in_t - cached_t
                        cost = ( (uncached_t * p_in) + (cached_t * (p_in * 0.5)) + (out_t * p_out) ) / 1000.0
                    else:
                        cost = ( (in_t * p_in) + (out_t * p_out) ) / 1000.0
                    cached_costs.append(cost)
                cost_mean = sum(cached_costs) / len(cached_costs)
            
            if (pd.isna(cost_mean) or cost_mean == 0.0) and "XGBoost" not in condition:
                # Fallback to standard base cost (~$2.51 per 100 loans)
                cost_mean = 0.0251
        cost_100 = cost_mean * 100.0
        
        tp = defaults * recall
        fn = defaults - tp
        total_rejections = tp / precision if precision > 0 else 0
        fp = total_rejections - tp
        tn = good_loans - fp
        
        loss_from_missed_defaults = fn * lgd_cost
        lost_profit_from_false_rejections = fp * EXPECTED_PROFIT
        api_cost = (cost_100 / 100.0) * PORTFOLIO_SIZE
        total_bank_cost = loss_from_missed_defaults + lost_profit_from_false_rejections + api_cost
        
        results.append({
            "Condition": condition,
            "Defaults Caught (TP)": f"{tp:.1f} ({recall*100:.1f}%)",
            "False Rejections (FP)": f"{fp:.1f}",
            "Credit Default Loss": loss_from_missed_defaults,
            "False Rejections Lost Profit": lost_profit_from_false_rejections,
            "API Token Cost": api_cost,
            "Total Cost": total_bank_cost
        })
        
    sim_df = pd.DataFrame(results).set_index("Condition")
    
    if baseline_condition_name and baseline_condition_name in sim_df.index:
        control_total = sim_df.loc[baseline_condition_name, "Total Cost"]
        sim_df["Net Financial Impact ($)"] = sim_df["Total Cost"] - control_total
        if control_total > 0:
            sim_df["Net Financial Impact (%)"] = (sim_df["Net Financial Impact ($)"] / control_total) * 100.0
        else:
            sim_df["Net Financial Impact (%)"] = 0.0
    else:
        sim_df["Net Financial Impact ($)"] = 0.0
        sim_df["Net Financial Impact (%)"] = 0.0
        
    formatted_df = sim_df.copy()
    formatted_df["Credit Default Loss"] = formatted_df["Credit Default Loss"].map("${:,.2f}".format)
    formatted_df["False Rejections Lost Profit"] = formatted_df["False Rejections Lost Profit"].map("${:,.2f}".format)
    formatted_df["API Token Cost"] = formatted_df["API Token Cost"].map("${:,.2f}".format)
    formatted_df["Total Cost"] = formatted_df["Total Cost"].map("${:,.2f}".format)
    formatted_df["Net Financial Impact ($)"] = formatted_df["Net Financial Impact ($)"].map(
        lambda x: f"+${x:,.2f}" if x > 0 else (f"-${abs(x):,.2f}" if x < 0 else "$0.00")
    )
    formatted_df["Net Financial Impact (%)"] = formatted_df["Net Financial Impact (%)"].map(
        lambda x: f"+{x:.2f}%" if x > 0 else (f"-{abs(x):.2f}%" if x < 0 else "0.00%")
    )
    return formatted_df

print("Cost simulation engine initialized.")

Cost simulation engine initialized.


## Part 1 — Credit Economics Ledgers

Run the cost model on the 02b prompt-variant metrics. This turns an accuracy or F1 gap between prompts into euros, and it shows how a variant that looks good on paper can still lose money by over-rejecting.

In [3]:
print("=== Phase 2: Prompt Engineering Financial Ledger ===")
if os.path.exists(METRICS_2B_PATH):
    phase2_df = run_financial_simulation(METRICS_2B_PATH, "XGBoost (structured)")
    display(phase2_df)
else:
    print(f"Metrics file not found at: {METRICS_2B_PATH}. Run 02b_Prompt_Variance.ipynb first.")

=== Phase 2: Prompt Engineering Financial Ledger ===
Injected baselines from Phase 1a: ['XGBoost (structured)', 'baseline']


,Defaults Caught (TP),False Rejections (FP),Credit Default Loss,False Rejections Lost Profit,API Token Cost,Total Cost,Net Financial Impact ($),Net Financial Impact (%)
Condition,,,,,,,,
XGBoost (structured),70.0 (46.7%),270.0,"$400,000.00","$540,000.00",$0.00,"$940,000.00",$0.00,0.00%
baseline,60.0 (40.0%),100.0,"$450,000.00","$200,000.00",$2.49,"$650,002.49","-$289,997.51",-30.85%
conservative,100.0 (66.7%),410.0,"$250,000.00","$820,000.00",$2.59,"$1,070,002.59","+$130,002.59",+13.83%
chain_of_thought,50.0 (33.3%),70.0,"$500,000.00","$140,000.00",$4.29,"$640,004.29","-$299,995.71",-31.91%
few_shot,50.0 (33.3%),80.0,"$500,000.00","$160,000.00",$2.49,"$660,002.49","-$279,997.51",-29.79%
top_features_only,70.0 (46.7%),240.0,"$400,000.00","$480,000.00",$4.85,"$880,004.85","-$59,995.15",-6.38%
structured_4factor,60.0 (40.0%),130.0,"$450,000.00","$260,000.00",$2.90,"$710,002.90","-$229,997.10",-24.47%
risk_signal_guide,60.0 (40.0%),160.0,"$450,000.00","$320,000.00",$3.26,"$770,003.26","-$169,996.74",-18.08%


## Part 2 — Qualitative Reasoning Fingerprinting

We sample 10 correct/incorrect reasoning chains for each prompt variant in Phase 2, then run the GPT-5.4 qualitative judge to identify feature anchors, speculation bias, or explainability flaws.

In [4]:
import random
import json

JUDGE_SYSTEM = (
    "You are reviewing how an AI credit risk model reasons about loan applications. "
    "Write a concise qualitative characterisation (4-6 sentences) of what is DISTINCTIVE "
    "about this prompt variant's decision reasoning style. Describe:\n"
    "- Which loan features or signals it consistently anchors on\n"
    "- Its overall risk posture (e.g. conservative, optimistic, balanced, formulaic)\n"
    "- Any systematic patterns, blind spots, or tendencies across samples\n"
    "- Whether its reasoning feels specific to each loan or generic and templated\n\n"
    "Do NOT score or rank. Do NOT say whether the model is good or bad. "
    "Just describe its reasoning fingerprint as you would describe a person's decision-making style."
)

qual_results = {}
out_path = f"{RESULTS_DIR}/llm/02c_qualitative_financial.json"
FORCE_RERUN_JUDGE = False  # Set to True to re-run all judge LLM calls

if not FORCE_RERUN_JUDGE and os.path.exists(out_path):
    print(f"Found existing qualitative analysis results at: {out_path}! Loading instead of calling OpenAI API...")
    with open(out_path, "r", encoding="utf-8") as f:
        qual_results = json.load(f)

def run_one_judge_task(idx, variant_name):
    df = pd.read_csv(PREDS_2B_PATH)
    sub = df[(df["phase"] == 1) & (df["variant"] == variant_name)]
    if sub.empty:
        return variant_name, "No predictions or reasoning logs found in metrics CSV."
    correct_df = sub[sub["correct"] == 1]
    incorrect_df = sub[sub["correct"] == 0]
    rng = random.Random(42)
    n_inc = min(5, len(incorrect_df))
    n_cor = min(10 - n_inc, len(correct_df))
    chosen_correct = correct_df.sample(n_cor, random_state=42) if len(correct_df) > 0 else pd.DataFrame()
    chosen_incorrect = incorrect_df.sample(n_inc, random_state=42) if len(incorrect_df) > 0 else pd.DataFrame()
    chosen = pd.concat([chosen_correct, chosen_incorrect])
    lines = []
    for r_idx, (_, r_row) in enumerate(chosen.iterrows(), 1):
        act_lbl = "Fully Paid" if r_row['actual'] == 1 else "Charged Off"
        pred_lbl = "Fully Paid" if r_row['prediction'] == 1 else "Charged Off"
        status = "correct" if r_row['actual'] == r_row['prediction'] else "WRONG"
        reasons_text = r_row.get('reasoning', '')
        lines.append(f"[{r_idx}] Actual: {act_lbl} | Predicted: {pred_lbl} ({status})\n"
                     f"    Reasoning: {reasons_text}")
    user_prompt = (
        f"Prompt Variant: {variant_name}\n"
        f"--- REASONING SAMPLES ---\n"
        + "\n\n".join(lines)
        + "\n--- END SAMPLES ---"
    )
    key_val = API_KEYS[idx % len(API_KEYS)]
    print(f"Judging {variant_name} on key {idx % len(API_KEYS) + 1}...")
    characterisation = call_llm(
        JUDGE_SYSTEM, user_prompt, 
        api_provider="openai", model="gpt-5.4", api_key=key_val
    )
    return variant_name, characterisation

if os.path.exists(PREDS_2B_PATH):
    df_preds = pd.read_csv(PREDS_2B_PATH)
    target_variants = df_preds[df_preds["phase"] == 1]["variant"].unique()
    
    # Identify missing variants to judge
    missing_variants = [v for v in target_variants if v not in qual_results]
    
    if missing_variants:
        print(f"Running LLM judge for {len(missing_variants)} missing prompt variants...")
        with ThreadPoolExecutor(max_workers=min(len(API_KEYS), 4)) as ex:
            futures = {ex.submit(run_one_judge_task, i, v): v for i, v in enumerate(missing_variants)}
            for fut in as_completed(futures):
                k, v = fut.result()
                qual_results[k] = v
                print(f"Finished judging: {k}")

        # Save qualitative findings to JSON
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(qual_results, f, indent=2, ensure_ascii=False)
        print(f"\nSaved qualitative + financial datasets to: {out_path}")
    else:
        print("\nAll prompt variants are already qualitatively characterized and loaded!")
else:
    print(f"Predictions file not found at: {PREDS_2B_PATH}. Run 02b_Prompt_Variance.ipynb first.")

Found existing qualitative analysis results at: ../../../data/results/llm/02c_qualitative_financial.json! Loading instead of calling OpenAI API...

All prompt variants are already qualitatively characterized and loaded!


In [5]:
# ── Pretty-print qualitative fingerprints ─────────────────────────────────
from IPython.display import display, Markdown

if qual_results:
    display(Markdown("---\n## Qualitative Reasoning Fingerprints"))
    for key, fingerprint in qual_results.items():
        formatted_fingerprint = fingerprint.replace('\n', '  \n> ')
        display(Markdown(
            f"### {key}\n"
            f"> {formatted_fingerprint}"
        ))
else:
    print("No qualitative results yet — run the judge cell first.")


---
## Qualitative Reasoning Fingerprints

### conservative
> This conservative variant anchors most heavily on classic downside credit signals: debt-to-income, revolving utilization, interest rate/LC grade, FICO band, prior delinquencies, and stability markers like income, employment length, and credit history. Its posture is clearly caution-first and asymmetrically sensitive to negative indicators, often treating subprime-leaning pricing, high utilization, or mixed profiles as sufficient reason to lean toward charge-off even when there are meaningful compensating strengths. A recurring pattern is that it explicitly frames decisions around “mixed signals” and then resolves ambiguity by defaulting to the riskier outcome, especially when leverage or pricing looks stretched. Its main blind spot is a tendency to overweight broad risk labels such as grade, interest rate, and utilization relative to stronger repayment-capacity evidence, though in a few cases it swings the other way when the file looks generally clean on surface credit metrics. The reasoning feels fairly structured and somewhat templated, but still grounded in the specific loan’s feature mix rather than being purely generic.

### chain_of_thought
> This chain-of-thought variant consistently anchors on a familiar credit stack: grade/interest rate, FICO, recent delinquency indicators, revolving utilization, DTI, income versus payment, and then secondary stability cues like homeownership, employment length, verification, and credit history length. Its posture is broadly balanced but slightly credit-bureau-trusting, often letting clean historical file signals and capacity markers outweigh softer warning signs, while becoming notably stricter when delinquencies are recent and explicitly repeated. Across samples, it uses an additive “risk signals vs protective factors” ledger, with strong weight on absence of bankruptcies/public records and long credit history, and it tends to interpret debt consolidation or manageable payment size as reassuring. A recurring tendency is to underreact to layered medium-risk combinations—especially long terms, weaker grades, elevated rates, and moderate delinquency/utilization—if income, FICO, and file cleanliness look respectable. The reasoning feels fairly specific to each loan because it cites concrete values and case details, but the structure is highly templated and formulaic, following the same compare-and-offset pattern almost every time.

### few_shot
> This few-shot variant reasons in a fairly checklist-driven way, anchoring most heavily on FICO band, LendingClub grade/interest rate, recent delinquencies, revolving utilization, income verification/strength, and the length of employment and credit history. Its posture is broadly balanced but slightly conservative when multiple risk flags stack up, especially high utilization plus weaker grade/rate plus delinquency history; at the same time, it will readily offset those negatives with “compensating factors” like solid income, clean bankruptcy/public-record history, and long tenure. A distinctive pattern is that it treats clean traditional credit signals as highly reassuring and often frames decisions as a tradeoff between a few negatives and an otherwise stable profile. It seems somewhat less sensitive to interactions among risk factors at the tails, sometimes overtrusting decent FICO/income when a loan later charges off, or overpenalizing very high rate/grade/utilization even when the loan is ultimately repaid. The language feels fairly specific to each loan’s feature mix, but the argumentative structure is templated: cite key positives, acknowledge a couple negatives, then conclude with an “overall profile” judgment.

### top_features_only
> This variant reasons in a stripped-down, feature-priority way, repeatedly anchoring on a small set of classic credit signals: grade/interest rate, FICO band, debt-to-income, revolving utilization, delinquencies/public records, income relative to loan size, and credit-history length. Its posture is fairly balanced but somewhat scorecard-like, often weighing a few strong positives against a few obvious negatives and then choosing the side that feels more dominant. A clear pattern is that it treats loan grade, term length, recent delinquency, and utilization as especially decisive on riskier files, while on safer-looking files it leans heavily on clean recent credit behavior and affordability cues. It tends to underuse richer contextual nuance beyond those top factors, so its mistakes often look like cases where a superficially clean profile overrides latent risk, or where a cluster of headline risk flags overwhelms otherwise stable repayment capacity. Overall, the reasoning feels somewhat templated and formulaic, with specific feature references for each loan but a very consistent “list positives, list negatives, then net them out” structure across samples.

### structured_4factor
> This variant has a distinctly four-bucket style: it repeatedly anchors on repayment capacity, credit quality/history, current debt stress, and loan structure, often explicitly balancing income vs. monthly payment, DTI, revolving utilization, FICO, delinquencies/public records, and grade/term. Its risk posture is fairly conservative and additive—negative signals like high utilization, recent delinquencies, bankruptcies, long 60-month terms, and weaker grades are treated as compounding risks that can outweigh otherwise solid income or long credit history. Across samples, it shows a systematic tendency to trust clean derogatory history and low DTI as strong positives, while heavily penalizing visible stress markers and structural loan risk, especially when multiple red flags appear together. A mild blind spot is that it can overreact to surface credit-risk flags even when broader repayment performance is stable, and it sometimes seems to underappreciate cases where good credit profile masks eventual default. The reasoning feels semi-templated rather than deeply bespoke: each case is framed as a structured tradeoff among the same recurring factors, with only modest tailoring to the specific loan.

### risk_signal_guide
> This variant reasons in a checklist-like way, anchoring most heavily on loan grade and interest rate, then on a recurring cluster of credit cleanliness signals: revolving utilization, DTI, delinquencies, bankruptcies/public records, income verification, and sometimes FICO, employment length, and credit history length. Its posture is fairly conservative and adverse-signal-driven, especially when it sees very high utilization or any derogatory records, which it often treats as decisive even when other parts of the file look stable. It also appears to carry some priors about loan purpose, tending to frame debt_consolidation and credit_card as relatively safer and medical as riskier. Across samples, it systematically balances positives and negatives in a formulaic “offset” structure, but it may overweight conspicuous red flags and underreact to more subtle combinations that still imply trouble. Overall, the reasoning feels somewhat templated rather than deeply case-specific: the same feature hierarchy and contrastive phrasing recur, with only modest tailoring to the individual loan.